# R3-01 · Introducción al NLP — Lección

**Bootcamp de Datos para Funcionarios Públicos · Formación Pública**
Línea R3 · *Especialista en IA Aplicada* · Semana 2 · Se ejecuta en Google Colab.

> 🚀 **Cómo se trabaja:** lee, ejecuta cada celda con **▶︎** (o `Shift`+`Enter`) y completa los `TODO`. Cada ejercicio termina en una **celda de chequeo** que muestra ✅ si está bien o una pista si todavía no.

---

## ¿Qué vas a lograr?

1. **Normalizar** y **tokenizar** texto en español, quitando *stopwords*.
2. Medir **frecuencias** y entender por qué no bastan.
3. Representar documentos con **TF-IDF**.
4. Entrenar y evaluar un **clasificador de temas** (train/test estratificado, línea base).
5. **Leer los errores** del clasificador con criterio.

**Competencia de salida:** dado un conjunto de textos del Estado, prepararlos, representarlos y clasificarlos, evaluando honestamente el resultado.

**Dato real:** 245 fichas de trámites de ChileAtiende con texto completo y su tema oficial (18 temas).

**Entregable:** que las **6 celdas de chequeo** muestren ✅.

In [ ]:
# ── Preparación del entorno (ejecuta esta celda primero) ──────────────────────
import os, urllib.request
import pandas as pd
import re, unicodedata
from collections import Counter

# "En vivo o caché": usa el archivo local; si falta (ej. Colab), lo baja del repo.
CSV = "fichas.csv"
if not os.path.exists(CSV):
    urllib.request.urlretrieve(f"https://raw.githubusercontent.com/formacionpublica-bootcamp/bootcamp/main/data/fichas.csv", CSV)

df = pd.read_csv(CSV)
print("Datos cargados:", df.shape, "filas x columnas")
df.head()

In [ ]:
# Palabras vacías (stopwords) del español, YA sin tildes (se comparan contra texto normalizado).
# Aparecen en cualquier texto y no dicen nada del tema. Lista provista para todo el módulo.
STOPWORDS = set("""
a al algo algunas algunos ante antes como con contra cual cuales cuando de del desde donde
durante e el ella ellas ellos en entre era es esa esas ese eso esos esta estan estar estas
este esto estos fue ha han hasta hay la las le les lo los mas me mi mis muy no nos o otra
otras otro otros para pero por porque puede pueden que quien quienes se sea segun ser si sin
sobre son su sus tambien te tiene tienen todo todos tu tus u un una uno unos usted y ya
""".split())
print(len(STOPWORDS), "stopwords. Ejemplo de texto real:")
print(df.loc[df["id"] == 39484, "texto"].iloc[0][:300], "...")

## Por qué el texto es un dato difícil

Hasta ahora tus datos fueron **números y categorías** en columnas. Pero buena parte de la
información del Estado es **texto libre**: fichas de trámites, oficios, reclamos, glosas. El
**NLP** (Procesamiento de Lenguaje Natural) enseña a la máquina a **leer texto a escala**.

Una máquina no «entiende» palabras: solo opera con números. Y el lenguaje es irregular:
`"Pensión"`, `"PENSION"` y `"pensión,"` son la misma palabra para ti y tres textos distintos
para el computador. Por eso todo proyecto de texto sigue la misma **tubería**:

1. **Normalizar:** minúsculas, sin tildes, sin signos.
2. **Tokenizar:** partir en unidades (palabras) llamadas *tokens*.
3. **Quitar stopwords:** palabras vacías (*de, la, que, para…*) que están en todo texto.
4. **Representar:** convertir cada texto en un vector de números (conteos, **TF-IDF**).
5. **Modelar:** con esos números, un modelo aprende a **clasificar** — aquí, el `tema` de cada ficha.

Trabajamos con las **245 fichas de ChileAtiende**. Cada una trae su `tema` asignado por
ChileAtiende (18 temas): eso permite entrenar un clasificador y **medir** cuánto acierta.

> Una analogía: normalizar y tokenizar es **ordenar la bodega** antes del inventario. Si los
> mismos productos están con etiquetas distintas, cualquier conteo te miente.

## 1. Normalizar el texto

Programa `normalizar(texto)`, en este orden: **minúsculas** → **quitar tildes** (descomponer con `unicodedata.normalize("NFKD", ...)` y descartar las marcas con `unicodedata.combining`) → reemplazar todo lo que no sea letra o número por espacio (`re.sub`) → colapsar espacios y recortar.

> ⚠️ La `ñ` también se descompone (n + virgulilla) y queda como `n`: «año» → «ano». Es un costo conocido de este método; para contar temas casi nunca importa.

### ✍️ Ejercicio 1 — Escribe `normalizar`

In [ ]:
def normalizar(texto):
    texto = texto.lower()
    # TODO: descompone con unicodedata.normalize("NFKD", texto)
    ...
    # TODO: descarta las marcas de acento: "".join(c for c in texto if not unicodedata.combining(c))
    ...
    # TODO: reemplaza todo lo que no sea a-z, 0-9 o espacio por " " (re.sub)
    ...
    texto = re.sub(r"\s+", " ", texto).strip()
    return texto

print(normalizar("Pensión Garantizada Universal (PGU): ¡$250.275!"))

In [ ]:
# ── Celda de chequeo · Ejercicio 1 ──────────────────────────────────────────
try:
    assert normalizar("Pensión Garantizada Universal (PGU): ¡$250.275!") == "pension garantizada universal pgu 250 275"
    assert normalizar("  BONO   de Invierno ") == "bono de invierno"
    assert normalizar("Á É Í Ó Ú ñ 2026!!") == "a e i o u n 2026"
    print("✅ Ejercicio 1: ¡correcto!")
except Exception as e:
    print("❌ Aún no. Revisa tu respuesta y vuelve a intentarlo.")
    print("   Pista:", "Tres líneas: texto = unicodedata.normalize('NFKD', texto); texto = ''.join(c for c in texto if not unicodedata.combining(c)); texto = re.sub(r'[^a-z0-9\\s]', ' ', texto).")
    print("   Detalle:", e)

## 2. Tokenizar y quitar palabras vacías

Programa `tokenizar(texto)`: normaliza (reutiliza tu función), parte por espacios y devuelve solo los tokens que **no** están en `STOPWORDS` **y** tienen **3 o más** caracteres.

### ✍️ Ejercicio 2 — Escribe `tokenizar`

In [ ]:
def tokenizar(texto):
    # TODO: comprensión sobre normalizar(texto).split() que descarte STOPWORDS y tokens de < 3 caracteres
    ...

print(tokenizar("Es un beneficio no postulable de $81.257 que recibirás junto con tu pensión de mayo"))

In [ ]:
# ── Celda de chequeo · Ejercicio 2 ──────────────────────────────────────────
try:
    assert tokenizar("Es un beneficio no postulable de $81.257") == ["beneficio", "postulable", "257"]
    assert tokenizar("Pensión de VEJEZ para la persona") == ["pension", "vejez", "persona"]
    assert "de" not in tokenizar("Certificado de nacimiento")
    print("✅ Ejercicio 2: ¡correcto!")
except Exception as e:
    print("❌ Aún no. Revisa tu respuesta y vuelve a intentarlo.")
    print("   Pista:", '[t for t in normalizar(texto).split() if t not in STOPWORDS and len(t) >= 3]')
    print("   Detalle:", e)

## 3. Frecuencias: ¿de qué hablan las fichas?

Con el texto tokenizado podemos **contar**. Junta los tokens de las 245 fichas en un `Counter` y mira las palabras más frecuentes. Pregúntate: ¿te dicen algo del **tema** de una ficha en particular?

### ✍️ Ejercicio 3 — Construye la bolsa de palabras del corpus

In [ ]:
frecuencias = Counter()
for texto in df["texto"]:
    # TODO: agrega los tokens de este texto al contador (frecuencias.update(...))
    ...
print(len(frecuencias), "palabras distintas")
print(frecuencias.most_common(10))

In [ ]:
# ── Celda de chequeo · Ejercicio 3 ──────────────────────────────────────────
try:
    _esperado = Counter()
    for _t in df["texto"]:
        _esperado.update([w for w in normalizar(_t).split() if w not in STOPWORDS and len(w) >= 3])
    assert isinstance(frecuencias, Counter) and len(frecuencias) > 0
    assert frecuencias.most_common(1) == _esperado.most_common(1)
    assert frecuencias["pension"] == _esperado["pension"]
    print("✅ Ejercicio 3: ¡correcto!")
except Exception as e:
    print("❌ Aún no. Revisa tu respuesta y vuelve a intentarlo.")
    print("   Pista:", 'Dentro del for: frecuencias.update(tokenizar(texto)).')
    print("   Detalle:", e)

## 4. TF-IDF: las palabras que distinguen

Las más frecuentes (*web, trámite, sitio, certificado*) vienen de instrucciones que se repiten en casi **todas** las fichas («ingresa al sitio web…»): no distinguen a ninguna. **TF-IDF** corrige eso: pesa cada palabra por cuánto aparece en *esta* ficha (**TF**) y la castiga si aparece en muchas fichas (**IDF**). Las palabras con TF-IDF alto son las que **caracterizan** a un documento.

`TfidfVectorizer` de scikit-learn hace el cálculo; le pasamos **tu** `tokenizar` como `analyzer` y `min_df=2` (ignora palabras que aparecen en una sola ficha). El resultado `X` es una matriz: una fila por ficha, una columna por palabra.

### ✍️ Ejercicio 4 — Vectoriza las fichas y lee el Bono de Invierno

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

# TODO: crea el vectorizador con analyzer=tokenizar y min_df=2
vectorizador = ...
# TODO: ajústalo y transforma df["texto"] en una sola llamada (fit_transform)
X = ...
terminos = vectorizador.get_feature_names_out()

i = df.index[df["id"] == 39484][0]            # fila del Bono de Invierno
pesos = X[i].toarray()[0]
# TODO: los 6 términos de mayor peso: terminos[pesos.argsort()[::-1][:6]] (envuelto en list)
top_tfidf = ...
print("matriz:", X.shape, "(fichas x palabras)")
print("TF-IDF más alto:", top_tfidf)

In [ ]:
# ── Celda de chequeo · Ejercicio 4 ──────────────────────────────────────────
try:
    assert X.shape[0] == len(df) and X.shape[1] == len(terminos)
    assert len(top_tfidf) == 6 and top_tfidf[0] == terminos[pesos.argmax()]
    assert "pension" in top_tfidf and "tramite" not in top_tfidf
    print("✅ Ejercicio 4: ¡correcto!")
except Exception as e:
    print("❌ Aún no. Revisa tu respuesta y vuelve a intentarlo.")
    print("   Pista:", "TfidfVectorizer(analyzer=tokenizar, min_df=2); X = vectorizador.fit_transform(df['texto']); top_tfidf = list(terminos[pesos.argsort()[::-1][:6]]).")
    print("   Detalle:", e)

## 5. Clasificar el tema desde el texto

Ahora el modelo: ¿puede una máquina adivinar el `tema` de una ficha leyendo solo su texto?

1. Separa las fichas en **entrenamiento** (75 %) y **prueba** (25 %) con `train_test_split(..., stratify=df["tema"], random_state=42)`. *Estratificar* mantiene la proporción de cada tema en ambos lados.
2. Ajusta un vectorizador **solo con entrenamiento** (`fit_transform`) y aplícalo a prueba (`transform`). Si el vectorizador «viera» las fichas de prueba, la evaluación quedaría contaminada.
3. Entrena una **regresión logística** y mide la **exactitud** (*accuracy*) en prueba.
4. Compárala con la **línea base**: adivinar siempre el tema más común.

### ✍️ Ejercicio 5 — Entrena y evalúa el clasificador

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

X_train_txt, X_test_txt, y_train, y_test = train_test_split(
    df["texto"], df["tema"], test_size=0.25, stratify=df["tema"], random_state=42)

vec = TfidfVectorizer(analyzer=tokenizar, min_df=2)
# TODO: fit_transform con el texto de ENTRENAMIENTO; solo transform con el de PRUEBA
X_train = ...
X_test = ...

modelo = LogisticRegression(max_iter=1000)
# TODO: entrena el modelo con X_train, y_train y predice sobre X_test
...
pred = ...

# TODO: exactitud en prueba (accuracy_score) y línea base (tema más común / total de prueba)
exactitud = ...
linea_base = ...
print("exactitud:", exactitud, "| línea base:", linea_base)

In [ ]:
# ── Celda de chequeo · Ejercicio 5 ──────────────────────────────────────────
try:
    assert len(pred) == len(y_test) == 62
    assert abs(exactitud - accuracy_score(y_test, pred)) < 1e-12
    assert abs(linea_base - y_test.value_counts().max() / len(y_test)) < 1e-12
    assert exactitud > 3 * linea_base, "el modelo debería superar con creces a la línea base"
    print("✅ Ejercicio 5: ¡correcto!")
except Exception as e:
    print("❌ Aún no. Revisa tu respuesta y vuelve a intentarlo.")
    print("   Pista:", 'X_train = vec.fit_transform(X_train_txt); X_test = vec.transform(X_test_txt); modelo.fit(X_train, y_train); pred = modelo.predict(X_test); accuracy_score(y_test, pred).')
    print("   Detalle:", e)

## 6. Leer los errores del clasificador

Una exactitud cercana al 56 % suena modesta, pero con 18 temas posibles es ~9 veces la línea base. Lo más valioso, sin embargo, está en **los errores**: arma una tabla con las fichas mal clasificadas y mira cuál es el tema que **más «atrae»** predicciones equivocadas.

### ✍️ Ejercicio 6 — Arma la tabla de errores

In [ ]:
tabla = pd.DataFrame({
    "titulo": df.loc[y_test.index, "titulo"],
    "real": y_test,
    "predicho": pred,
})
# TODO: filas donde el tema real es distinto del predicho
errores = ...
n_errores = ...
# TODO: el tema que más aparece como "predicho" dentro de los errores (value_counts().idxmax())
tema_iman = ...
print(n_errores, "errores | tema que más atrae errores:", tema_iman)

In [ ]:
# ── Celda de chequeo · Ejercicio 6 ──────────────────────────────────────────
try:
    assert n_errores == int((y_test.values != pred).sum())
    assert set(errores.columns) == {"titulo", "real", "predicho"}
    assert (errores["real"] != errores["predicho"]).all()
    assert tema_iman == errores["predicho"].value_counts().idxmax()
    print("✅ Ejercicio 6: ¡correcto!")
except Exception as e:
    print("❌ Aún no. Revisa tu respuesta y vuelve a intentarlo.")
    print("   Pista:", 'errores = tabla[tabla["real"] != tabla["predicho"]]; n_errores = len(errores); errores["predicho"].value_counts().idxmax().')
    print("   Detalle:", e)

## ✍️ Reflexión (no se corrige, pero es lo más importante)

Mira la tabla de arriba: el modelo mandó a *trabajo-y-cesantía* fichas como **Aportes del
empleador al Sistema de Pensiones** (real: jubilación), **Beneficios asociados a la ley Bono
Laboral** (real: bonos) o **apelar el porcentaje de incapacidad por accidente del trabajo**
(real: previsión). Algo parecido ocurre con el propio **Bono de Invierno** (real: jubilación;
predicho: bonos-y-beneficios-del-estado) y con la *Beca Residencia Indígena* (real:
pueblos-originarios; predicho: becas-y-créditos).

En una o dos frases: ¿son **errores del modelo** o señales de que una ficha puede pertenecer
legítimamente a **más de un tema**? Si este clasificador se usara para sugerir el tema de las
fichas nuevas, ¿lo dejarías decidir solo, o como **sugerencia** que revisa una persona?

> Leer los errores — no solo la exactitud — es lo que separa un modelo útil de uno peligroso.

## Errores típicos

- **Vectorizar antes de separar** → el vectorizador «ve» las fichas de prueba (fuga de información) y la exactitud sale optimista.
- **`ValueError` al estratificar** → algún tema tiene un solo ejemplo; no se puede repartir entre entrenamiento y prueba.
- **Tokens raros como `257`** → los números sobreviven a la normalización; a veces conviene filtrarlos.
- **Mirar solo la exactitud** → esconde qué temas fallan y por qué (lo verás en la profundización).

## Cierre

- La tubería: **normalizar → tokenizar → stopwords → representar (TF-IDF) → modelar**.
- Las palabras **más frecuentes** no distinguen; **TF-IDF** destaca las que caracterizan a cada documento.
- Un clasificador lineal sobre TF-IDF adivina el tema de una ficha ~9 veces mejor que la línea base.
- Sus **errores** muestran el límite del método: cuenta palabras, no entiende — y una etiqueta única no siempre describe un documento.

> *En R3-02 darás el salto a un **modelo de lenguaje (LLM)** que sí capta significado: le pedirás
> que resuma una ficha en lenguaje claro y que extraiga sus requisitos como datos estructurados.*